# Lab 06 — Convolutional Neural Networks for Text Classification

**NLP Applications Laboratory | KLEF | 2026**

*From averaging word vectors to sliding filters over them — how a small architectural change lets a neural network learn what specific word combinations mean.*

## Objective

In Lab 05 we saw that averaging Word2Vec vectors underperformed even TF-IDF (81% vs 85%). We diagnosed the problem: **averaging destroys word order and dilutes distinctive signals**.

Sentence-BERT solved this with attention (88.45%), but SBERT is a transformer — which we haven't studied yet. Before jumping to transformers, we visit an intermediate architecture from 2014 that also fixes averaging without needing attention:

**Convolutional Neural Networks for text.**

A CNN slides small filters over sequences of word vectors, learning to detect specific n-gram patterns. Where averaging says "here are all the words, blended together", a CNN says "let me look for meaningful 3-word combinations, then keep the strongest match I find." The word-order information survives.

By the end of this lab, you will:

1. Understand **how a text CNN differs from an image CNN** — same math, different geometry.
2. Build a **tiny 4-filter CNN** on a hand-crafted 48-sentence corpus and see, literally by inspection, what each filter learned.
3. Trace **one specific sentence** through the network position by position — see which n-gram won max-pooling.
4. Scale the same architecture to **20 Newsgroups** and compare it against Lab 05's five methods.
5. Understand why CNNs bridged the gap between simple word-averaging and modern attention-based models.

The core question of this lab:

> *When you let a neural network learn its own filters instead of averaging pre-trained vectors, what does it actually learn — and how do we know?*

We will not just get an accuracy number. We will look **inside** the network and see the filters make sense.

## What you will learn before we start

This lab has **eleven phases**, organized as "understand the mechanism on a toy corpus, then apply to real data."

### Phases 1-8 — The visualizable half

**Phase 1** — Concept: why CNNs for text? What problem they solve that averaging doesn't.

**Phase 2** — Build the tiny corpus: 12 sentences × 4 classes = 48 sentences. Sentiment × topic hybrid (positive/negative × food/tech) so filters can learn either dimension.

**Phase 3** — Convert sentences to sequences of Word2Vec vectors. This is what CNNs read.

**Phase 4** — Anatomy of a single convolutional filter. What "convolution over words" actually computes.

**Phase 5** — Build a tiny 4-filter CNN in PyTorch. Small enough to inspect every parameter.

**Phase 6** — Train on the 48-sentence corpus. Should reach near-perfect accuracy quickly — the point isn't difficulty, it's visibility.

**Phase 7** — Visualize what the filters learned. Three inspections:
- Which n-grams activate each filter most strongly?
- Trace one sentence position-by-position through the network
- Which filters push toward which classes?

**Phase 8** — Chat with Qwen about the mechanism.

### Phases 9-11 — The real-world half

**Phase 9** — Scale up to 20 Newsgroups. Same architecture, more filters, longer training.

**Phase 10** — Grand comparison. Add the CNN row to Lab 05's table. Where does it land between averaged Word2Vec (81%) and Sentence-BERT (88%)?

**Phase 11** — Wrap-up + post-lab assignment.

### One honest note about scale and difficulty

The tiny corpus is deliberately easy — a well-trained CNN should classify all 48 sentences perfectly. The point is not to challenge the model but to make its learned behavior human-inspectable. On 20 Newsgroups we lose that inspectability but gain realistic evaluation.

This "small then large" pattern is standard in ML research: **prototype on data where you can see the mechanism, deploy on data where you can measure the performance**.

---

## Phase 1 — Why CNNs for text?

### The problem we're solving

In Lab 05 we tried two approaches:

- **Average Word2Vec vectors** → lost word order, got 81% on 20 Newsgroups.
- **Sentence-BERT** (transformer with attention) → preserved order, got 88%.

Between "throw away word order" and "full transformer" there was a gap. CNNs fill that gap.

### The core insight (Kim, 2014)

Imagine you're classifying restaurant reviews. What tells you a review is positive?

Not individual words like `good` or `bad` alone — those appear in reviews of both polarities ("not good", "really bad but the fries were good"). What matters is **short phrases**:

- `"absolutely delicious"` → positive
- `"never coming back"` → negative
- `"waited forever"` → negative
- `"cooked perfectly"` → positive

These are **local n-gram patterns**. If a model could learn to detect them, it would classify reviews correctly.

**This is exactly what CNNs do.** A CNN filter of width 3 slides over the text and computes a score at every position. A well-trained filter has weights that "light up" on specific patterns like `"absolutely delicious"` and stay quiet elsewhere. Max-pooling then asks: "did this pattern occur anywhere in the sentence?"

### Text CNN vs image CNN — same math, different geometry

The math is identical. The geometry differs:

| | Image CNN | Text CNN |
|---|---|---|
| Input | 2D grid of pixels (H × W × channels) | 2D grid of word vectors (seq_len × embed_dim) |
| Filter shape | 2D (e.g., 3×3) | 2D (e.g., 3 × embed_dim) |
| Filter slides in | 2D (across height AND width) | 1D (across sequence only) |
| What a filter learns | Visual patterns (edges, textures) | Word-combination patterns (phrases) |
| Pooling | 2D max-pool over spatial regions | 1D max-pool over sequence positions |

The key difference: a text filter is as wide as the embedding (300 dims for Word2Vec). It doesn't slide *across* the embedding dimension — it only slides *along the sequence*. Each filter sees a small window of consecutive words, computes one score per position.

### What we'll build

A minimal architecture that shows this in action:


Small, clean, fully inspectable. In Phase 7 we'll open up each filter and see what pattern it learned.

### Why we're doing this before transformers

Two reasons:

1. **CNNs still ship in production.** For latency-sensitive tasks like spam filtering, content moderation, and edge-device NLP, CNNs remain competitive with transformers because they're 10-100x faster to run.

2. **Understanding CNN limits motivates transformers.** After this lab you will know exactly what a CNN can and cannot do. When we introduce transformers next, you will understand precisely which of those limits attention solves.

---

## Phase 2 — The tiny corpus

To *see* what a CNN learns, we need a corpus small enough to trace by hand but structured enough that different filters can learn different patterns.

We use a **sentiment × topic hybrid** — 4 classes formed by combining two dimensions:

| | Food | Tech |
|---|---|---|
| **Positive** | `positive_food` | `positive_tech` |
| **Negative** | `negative_food` | `negative_tech` |

12 sentences per class × 4 classes = **48 sentences total**. Each sentence is 3-6 words. Every sentence is a real short opinion.

### Why this cross-cutting structure matters

After training, we expect filters to specialize in one of three ways:

- **Sentiment filters** — fire on positive OR negative vocabulary, regardless of topic (`amazing`, `perfect` vs `terrible`, `broken`)
- **Topic filters** — fire on food OR tech vocabulary, regardless of sentiment (`pizza`, `bread` vs `laptop`, `phone`)
- **Combination filters** — fire on specific class-defining phrases (`love pizza`, `phone crashed`)

If the corpus had only 4 unrelated topics with no cross-cutting structure, students would only see topic filters emerge — a less interesting story. The sentiment × topic hybrid forces filters to specialize along either dimension, giving richer visualization possibilities in Phase 7.

In [1]:
# 48 sentences, 12 per class, sentiment × topic hybrid
tiny_corpus = {
    'positive_food': [
        "I love eating pizza", "This bread tastes amazing", "Pasta is delicious",
        "The pizza was perfect", "I enjoyed the pasta", "Fresh bread is wonderful",
        "This meal is fantastic", "The pizza tastes great", "I love hot pasta",
        "Delicious homemade bread", "The food was excellent", "Amazing pizza tonight",
    ],
    'negative_food': [
        "I hate cold pizza", "This bread is stale", "The pasta was terrible",
        "Awful burnt pizza", "The bread tastes bad", "I hated the pasta",
        "This meal is disgusting", "The pizza was overcooked", "Stale hard bread",
        "Terrible undercooked pasta", "The food was awful", "Disappointing pizza tonight",
    ],
    'positive_tech': [
        "My laptop is great", "This phone works perfectly", "The software is amazing",
        "I love my laptop", "This phone is fantastic", "The app works wonderfully",
        "Excellent phone camera", "My laptop runs fast", "The software is brilliant",
        "This phone feels premium", "I enjoy this app", "Amazing new laptop",
    ],
    'negative_tech': [
        "My laptop is broken", "This phone is terrible", "The software crashed",
        "I hate this laptop", "This phone stopped working", "The app keeps freezing",
        "Awful phone battery", "My laptop runs slowly", "The software is buggy",
        "This phone feels cheap", "I regret this app", "Broken new laptop",
    ],
}

# Flatten to (sentence, label_index) pairs
tiny_labels = sorted(tiny_corpus.keys())
tiny_label2idx = {lbl: i for i, lbl in enumerate(tiny_labels)}
tiny_data = [(s, tiny_label2idx[lbl]) for lbl in tiny_labels for s in tiny_corpus[lbl]]
print(f"Total sentences: {len(tiny_data)}")
print(f"Classes ({len(tiny_labels)}): {tiny_labels}")
print(f"\nSample sentences:")
for i in [0, 12, 24, 36]:
    print(f"  [{tiny_labels[tiny_data[i][1]]}] {tiny_data[i][0]}")

Total sentences: 48
Classes (4): ['negative_food', 'negative_tech', 'positive_food', 'positive_tech']

Sample sentences:
  [negative_food] I hate cold pizza
  [negative_tech] My laptop is broken
  [positive_food] I love eating pizza
  [positive_tech] My laptop is great


---

## Phase 3 — Represent sentences as sequences of Word2Vec vectors

Before a CNN can convolve over text, we need to convert each sentence into the right shape: a **matrix of shape (sequence_length, embedding_dimension)**.

For our tiny corpus with sentences of 3-6 words and 300-dim Word2Vec vectors, each sentence becomes a matrix of shape `(N, 300)` where N is the number of words.

### The two problems we need to solve

**Problem 1 — Missing words.** Word2Vec Google News doesn't have every English word. If a word from our corpus is not in the vocabulary, we need a fallback. We'll skip missing words silently.

**Problem 2 — Variable sentence length.** "I love pizza" is 3 words. "This bread tastes amazing" is 4 words. Our CNN needs a fixed input size to process sentences in batches. We'll **pad** shorter sentences with zero vectors to match the longest sentence in the corpus.

Padding with zeros is a standard trick — the zero vector has no direction, so it contributes nothing to any filter's output. A well-trained CNN learns to ignore padded positions.

### The pipeline for this phase

1. Reload Word2Vec (making the notebook self-contained)
2. Determine max sentence length in our corpus
3. Convert every sentence to a padded (max_len × 300) matrix
4. Stack all sentence matrices into one big tensor: (48, max_len, 300)

This tensor is what our CNN will consume.

# Reload Word2Vec

In [2]:
# Reload pretrained Word2Vec (Google News 300d) — cached from setup_data.py
import gensim.downloader as api
print("Loading Word2Vec from cache...")
w2v = api.load('word2vec-google-news-300')
print(f"Loaded. Vocab: {len(w2v.key_to_index):,} words | Dim: {w2v.vector_size}")

Loading Word2Vec from cache...
Loaded. Vocab: 3,000,000 words | Dim: 300


# Vocabulary coverage check

In [3]:
# Which words from our tiny corpus are in Word2Vec's vocabulary?
all_words = set()
for s, _ in tiny_data:
    for w in s.lower().split():
        all_words.add(w)

in_vocab = [w for w in sorted(all_words) if w in w2v]
out_vocab = [w for w in sorted(all_words) if w not in w2v]

print(f"Total unique words in tiny corpus: {len(all_words)}")
print(f"In Word2Vec vocab                : {len(in_vocab)}")
print(f"Missing from vocab               : {len(out_vocab)}")
if out_vocab:
    print(f"\nMissing words: {out_vocab}")

Total unique words in tiny corpus: 65
In Word2Vec vocab                : 65
Missing from vocab               : 0


# Determine padding length and convert sentences to matrices

In [4]:
# Find the longest sentence in our corpus — that's our padding target
import numpy as np
tokenized = [s.lower().split() for s, _ in tiny_data]
max_len = max(len(t) for t in tokenized)
print(f"Longest sentence: {max_len} words")
print(f"All sentence lengths: {sorted(set(len(t) for t in tokenized))}")

Longest sentence: 4 words
All sentence lengths: [3, 4]


In [5]:
# Convert every sentence to a (max_len × 300) matrix, padding with zeros
def sentence_to_matrix(sentence, max_len=max_len, dim=300):
    tokens = sentence.lower().split()
    matrix = np.zeros((max_len, dim), dtype=np.float32)
    for i, tok in enumerate(tokens[:max_len]):
        if tok in w2v:
            matrix[i] = w2v[tok]
    return matrix

X_tiny = np.stack([sentence_to_matrix(s) for s, _ in tiny_data])
y_tiny = np.array([lbl for _, lbl in tiny_data])
print(f"Input tensor X_tiny shape:  {X_tiny.shape}  (sentences × positions × dims)")
print(f"Label tensor y_tiny shape: {y_tiny.shape}")

Input tensor X_tiny shape:  (48, 4, 300)  (sentences × positions × dims)
Label tensor y_tiny shape: (48,)


In [6]:
# Inspect one padded sentence to confirm zeros where we expect them
example_idx = 0  # "I hate cold pizza" — 4 words
print(f"Sentence: {tiny_data[example_idx][0]!r}")
print(f"Its matrix shape: {X_tiny[example_idx].shape}")
print(f"Norm of each row: {[float(f'{np.linalg.norm(row):.2f}') for row in X_tiny[example_idx]]}")

Sentence: 'I hate cold pizza'
Its matrix shape: (4, 300)
Norm of each row: [2.63, 2.82, 2.73, 3.19]


# One extra small check to see padding in action

In [7]:
# Pick a 3-word sentence to see the padded position
for i, (s, _) in enumerate(tiny_data):
    if len(s.split()) == 3:
        print(f"Found 3-word sentence at index {i}: {s!r}")
        print(f"Norm of each row: {[float(f'{np.linalg.norm(row):.2f}') for row in X_tiny[i]]}")
        break

Found 3-word sentence at index 3: 'Awful burnt pizza'
Norm of each row: [2.61, 3.33, 3.19, 0.0]


---

## Phase 4 — Anatomy of a text convolution

Before we build a real CNN in Phase 5, let's understand exactly what one filter does mathematically. This is the "concept before code" cell.

### Setup

A single filter has:
- **Width** `k` — how many consecutive words it looks at (e.g., k=3 means "look at 3-word windows")
- **Depth** = the embedding dimension (300 for our Word2Vec)
- **A bias term** (a single number added at the end)

So a width-3 filter has shape `(3, 300)` — a small matrix of 900 numbers, plus 1 bias = 901 trainable parameters.

### What the filter does at each position

Given a sentence matrix of shape `(4, 300)` and a width-3 filter:

- Position 1: look at rows 1, 2, 3. Compute the sum of element-wise products between the filter and this 3-word window. Add bias. Apply an activation function (ReLU). Result: **one number**.
- Position 2: look at rows 2, 3, 4. Same computation. Result: **one number**.
- No more positions — the filter has slid off the end.

The filter produces a **1D score map** of length `sequence_length - filter_width + 1 = 4 - 3 + 1 = 2`.

### One picture

Sentence matrix (4 × 300):
Row 1: [word1_vector] ─┐
Row 2: [word2_vector] ─┤─→ Filter reads rows 1,2,3 → score for position 1
Row 3: [word3_vector] ─┤
─┘
Row 2: [word2_vector] ─┐
Row 3: [word3_vector] ─┤─→ Filter reads rows 2,3,4 → score for position 2
Row 4: [word4_vector] ─┤
─┘

Filter output: [score_pos1, score_pos2] (2 numbers)


### Then: max-pooling

After all filters have produced their score maps, we take the **maximum** score from each filter's output. This asks:

> "Regardless of *where* in the sentence, did this filter's target pattern occur? Give me its strongest match."

For our width-3 filter with 2 output positions: max([score_pos1, score_pos2]) = one number.

If we have 4 filters, max-pooling produces a **feature vector of length 4** — one number per filter.

### The whole tiny CNN in one diagram

Sentence: "I love eating pizza"
↓ (Word2Vec lookup + padding)
Matrix: (4 × 300)
↓ (4 filters of width 3 slide over sequence)
Score maps: 4 filters × 2 positions each
↓ (max-pool: take max per filter)
Feature vector: (4,)
↓ (linear layer: 4 → 4 classes)
Class logits: (4,)
↓ (softmax + argmax)
Prediction: "positive_food" (class 2)

# Build the tiny CNN in PyTorch

In [9]:
# The tiny CNN: 4 filters of width 3, max-pool, linear classifier
import torch
import torch.nn as nn

class TinyTextCNN(nn.Module):
    def __init__(self, embed_dim=300, n_filters=4, filter_width=3, n_classes=4):
        super().__init__()
        # Conv1d expects (batch, channels_in, length) where channels_in = embed_dim
        self.conv = nn.Conv1d(in_channels=embed_dim,
                              out_channels=n_filters,
                              kernel_size=filter_width)
        self.classifier = nn.Linear(n_filters, n_classes)
    def forward(self, x):
        # x arrives as (batch, seq_len, embed_dim). Conv1d wants (batch, embed_dim, seq_len).
        x = x.transpose(1, 2)                    # → (batch, embed_dim, seq_len)
        scores = torch.relu(self.conv(x))         # → (batch, n_filters, seq_len - filter_width + 1)
        pooled, _ = scores.max(dim=2)             # → (batch, n_filters) — max over positions
        return self.classifier(pooled)            # → (batch, n_classes)

# Instantiate the model

In [10]:
# Create the model with default settings
torch.manual_seed(42)                    # reproducibility
model = TinyTextCNN(embed_dim=300, n_filters=4, filter_width=3, n_classes=4)
print(model)

TinyTextCNN(
  (conv): Conv1d(300, 4, kernel_size=(3,), stride=(1,))
  (classifier): Linear(in_features=4, out_features=4, bias=True)
)


# Full architecture summary

In [11]:
# Detailed architecture summary: layer, input shape, output shape, parameters
def summarize(model, input_shape=(1, 4, 300)):
    print(f"{'Layer':<25} {'Input shape':<22} {'Output shape':<22} {'Params':>10}")
    print("=" * 80)
    x = torch.zeros(input_shape)
    print(f"{'INPUT (sentence tensor)':<25} {'—':<22} {str(tuple(x.shape)):<22} {'0':>10}")
    print(f"{'transpose(1,2)':<25} {str(tuple(x.shape)):<22} ", end='')
    x = x.transpose(1, 2)
    print(f"{str(tuple(x.shape)):<22} {'0':>10}")

    print(f"{'Conv1d(300, 4, k=3)':<25} {str(tuple(x.shape)):<22} ", end='')
    conv_out = model.conv(x)
    conv_params = sum(p.numel() for p in model.conv.parameters())
    print(f"{str(tuple(conv_out.shape)):<22} {conv_params:>10,}")

    print(f"{'ReLU':<25} {str(tuple(conv_out.shape)):<22} ", end='')
    relu_out = torch.relu(conv_out)
    print(f"{str(tuple(relu_out.shape)):<22} {'0':>10}")

    print(f"{'max-pool over positions':<25} {str(tuple(relu_out.shape)):<22} ", end='')
    pooled, _ = relu_out.max(dim=2)
    print(f"{str(tuple(pooled.shape)):<22} {'0':>10}")

    print(f"{'Linear(4, 4)':<25} {str(tuple(pooled.shape)):<22} ", end='')
    out = model.classifier(pooled)
    clf_params = sum(p.numel() for p in model.classifier.parameters())
    print(f"{str(tuple(out.shape)):<22} {clf_params:>10,}")

    total = sum(p.numel() for p in model.parameters())
    print("=" * 80)
    print(f"{'TOTAL TRAINABLE PARAMETERS':<69} {total:>10,}")

summarize(model)

Layer                     Input shape            Output shape               Params
INPUT (sentence tensor)   —                      (1, 4, 300)                     0
transpose(1,2)            (1, 4, 300)            (1, 300, 4)                     0
Conv1d(300, 4, k=3)       (1, 300, 4)            (1, 4, 2)                   3,604
ReLU                      (1, 4, 2)              (1, 4, 2)                       0
max-pool over positions   (1, 4, 2)              (1, 4)                          0
Linear(4, 4)              (1, 4)                 (1, 4)                         20
TOTAL TRAINABLE PARAMETERS                                                 3,624


# Set up training tensors, loss, and optimizer

In [12]:
# Convert numpy arrays to PyTorch tensors, set up loss and optimizer
X_tiny_t = torch.tensor(X_tiny, dtype=torch.float32)
y_tiny_t = torch.tensor(y_tiny, dtype=torch.long)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
print(f"X tensor shape: {X_tiny_t.shape}, dtype: {X_tiny_t.dtype}")
print(f"y tensor shape: {y_tiny_t.shape}, dtype: {y_tiny_t.dtype}")

X tensor shape: torch.Size([48, 4, 300]), dtype: torch.float32
y tensor shape: torch.Size([48]), dtype: torch.int64


# The training loop

In [13]:
# Full-batch training on 48 sentences for a few hundred epochs
EPOCHS = 300
losses, accuracies = [], []
for epoch in range(EPOCHS):
    optimizer.zero_grad()                     # 1. clear old gradients
    logits = model(X_tiny_t)                  # 2. forward pass
    loss = loss_fn(logits, y_tiny_t)          # 3. compute loss
    loss.backward()                            # 4. compute gradients
    optimizer.step()                           # 5. update parameters
    with torch.no_grad():
        acc = (logits.argmax(dim=1) == y_tiny_t).float().mean().item()
    losses.append(loss.item()); accuracies.append(acc)
    if epoch % 50 == 0 or epoch == EPOCHS - 1:
        print(f"Epoch {epoch:>3}  loss = {loss.item():.4f}  train_acc = {acc:.4f}")

Epoch   0  loss = 1.4081  train_acc = 0.2500
Epoch  50  loss = 0.0374  train_acc = 1.0000
Epoch 100  loss = 0.0043  train_acc = 1.0000
Epoch 150  loss = 0.0022  train_acc = 1.0000
Epoch 200  loss = 0.0014  train_acc = 1.0000
Epoch 250  loss = 0.0010  train_acc = 1.0000
Epoch 299  loss = 0.0007  train_acc = 1.0000


---

## Phase 7 — Visualize what the filters learned

Our CNN has 4 filters. Each filter, after training, has learned to look for a specific 3-word pattern. The question is: what pattern?

There are three ways to answer this:

### Method 1 — Top activating n-grams per filter

For each filter, run every possible 3-word window from the corpus through it. Rank the windows by activation score. Read the top-3 highest-scoring windows for each filter. If the corpus is well-structured (which ours is), the top n-grams for each filter will be human-interpretable.

**We expect to see**: some filters fire on positive vocabulary, some on negative, some on food, some on tech.

### Method 2 — Trace one sentence position by position

Pick one specific sentence. Compute the score at every position for every filter. See which filter's peak wins max-pooling. This shows exactly *how* the network made its decision on that sentence.

### Method 3 — Read the classifier weights

The classifier is a `Linear(4, 4)` layer — 4 filter features going to 4 class predictions. This means each class has a weight for each filter, telling us which filters "vote" for which classes.

If Filter 3 has a large positive weight for the `positive_food` class, it means "when Filter 3 fires strongly, the model tends to predict positive_food."

Together these three views tell a complete story: **what** each filter detects (Method 1), **how** they act on a specific input (Method 2), and **which classes** they support (Method 3).

Let's go through them one at a time.

# Method 1: top-activating n-grams for each filter

In [14]:
# For each filter, find which 3-word windows in the corpus activate it most strongly
model.eval()
with torch.no_grad():
    x = X_tiny_t.transpose(1, 2)                          # (48, 300, 4)
    all_scores = torch.relu(model.conv(x))                # (48, n_filters, 2 positions)

# For each filter, collect (score, sentence_idx, position) triples across all sentences
n_filters = all_scores.shape[1]
for f in range(n_filters):
    triples = []
    for sent_idx in range(len(tiny_data)):
        for pos in range(all_scores.shape[2]):
            score = all_scores[sent_idx, f, pos].item()
            triples.append((score, sent_idx, pos))
    triples.sort(reverse=True)
    print(f"\nFilter {f} — top 5 activating 3-word windows:")
    for score, sent_idx, pos in triples[:5]:
        tokens = tiny_data[sent_idx][0].lower().split()
        window = ' '.join(tokens[pos:pos+3])
        cls = tiny_labels[tiny_data[sent_idx][1]]
        print(f"  {score:>6.3f}  '{window}'  [from '{tiny_data[sent_idx][0]}' — {cls}]")


Filter 0 — top 5 activating 3-word windows:
  18.956  'awful phone battery'  [from 'Awful phone battery' — negative_tech]
  17.293  'my laptop is'  [from 'My laptop is great' — positive_tech]
  17.293  'my laptop is'  [from 'My laptop is broken' — negative_tech]
  16.753  'the app keeps'  [from 'The app keeps freezing' — negative_tech]
  16.582  'regret this app'  [from 'I regret this app' — negative_tech]

Filter 1 — top 5 activating 3-word windows:
   0.000  'new laptop'  [from 'Amazing new laptop' — positive_tech]
   0.000  'amazing new laptop'  [from 'Amazing new laptop' — positive_tech]
   0.000  'enjoy this app'  [from 'I enjoy this app' — positive_tech]
   0.000  'i enjoy this'  [from 'I enjoy this app' — positive_tech]
   0.000  'phone feels premium'  [from 'This phone feels premium' — positive_tech]

Filter 2 — top 5 activating 3-word windows:
  14.006  'undercooked pasta'  [from 'Terrible undercooked pasta' — negative_food]
  12.943  'disappointing pizza tonight'  [from 'Dis

Reading what your specific CNN learned

Filter 0 — "Tech" detector. All top-5 windows come from tech sentences (both positive and negative). It fires on both positive_tech ("my laptop is great") and negative_tech ("my laptop is broken", "awful phone battery"). This filter learned topic, not sentiment — it detects "this sentence is about a device/app" regardless of polarity.

Filter 1 — DEAD FILTER. Every activation is exactly 0.000. This filter never fires on anything. It contributes zero information to the classifier. This is a real and important phenomenon in neural network training called a dead ReLU — and it's worth teaching, not hiding.

Filter 2 — "Negative food" detector. All top-5 windows come from negative_food sentences (undercooked pasta, bad bread, disappointing pizza). This filter learned a combined topic × sentiment pattern: negative sentiment about food.

Filter 3 — "Positive food" detector. All top-5 windows come from positive_food sentences (pizza tastes great, meal is fantastic, bread tastes amazing). This filter learned the mirror image of Filter 2: positive sentiment about food.

The dead filter is a real learning moment

Filter 1 outputting all zeros isn't a mistake in your notebook — it's a known failure mode of the ReLU activation function. Here's what happened:

At initialization, Filter 1 got random weights that produced negative pre-activation values for most inputs.
ReLU clamped those negative values to 0.
With output stuck at 0, no gradient flowed back through Filter 1.
With no gradient, the filter's weights never got updated.
Filter 1 stayed dead throughout all 300 epochs.

Meanwhile, Filters 0, 2, 3 handled the classification just fine — model reached 100% accuracy on the tiny corpus. So Filter 1 was redundant, and its death didn't hurt accuracy. But those 900 parameters are wasted.

In production this is a common problem. Practitioners fix it by:

Using Leaky ReLU instead of ReLU (allows small negative activations, keeps gradients flowing)
Better weight initialization (Kaiming/He init specifically designed for ReLU)
More filters (redundancy so dead ones don't matter)
Batch normalization

This is a genuinely valuable thing for students to see. Real neural networks have dead neurons. Understanding why and how to detect them is a real skill.
What your specific filters tell us

The classifier now has three working feature detectors:

Filter 0: "is this tech?" (topic)
Filter 2: "is this negative food?" (topic × sentiment combined)
Filter 3: "is this positive food?" (topic × sentiment combined)

The classifier is doing something clever with just these three features: for the two food classes, Filters 2 and 3 provide direct evidence. For the two tech classes, Filter 0 signals "this is tech" but doesn't distinguish positive vs negative — so the classifier must use Filter 0's activation strength or negative activations in Filters 2 and 3 to figure out tech sentiment. We'll see exactly how in Method 3 when we read the classifier weights.

# Method 2: trace one sentence through the network

In [15]:
# Pick one sentence and trace it position-by-position through every filter
test_sentence_idx = 12  # a positive_food sentence  
sent, true_label = tiny_data[test_sentence_idx]
print(f"Tracing sentence: '{sent}'  (true class: {tiny_labels[true_label]})\n")

model.eval()
with torch.no_grad():
    x = X_tiny_t[test_sentence_idx:test_sentence_idx+1].transpose(1, 2)
    scores = torch.relu(model.conv(x))                           # (1, 4 filters, 2 positions)
    pooled, winning_positions = scores.max(dim=2)               # max over positions
    logits = model.classifier(pooled)
    predicted = logits.argmax(dim=1).item()

tokens = sent.lower().split()
print(f"{'Filter':<8} {'Pos 0 (words 0-2)':<28} {'Pos 1 (words 1-3)':<28} {'Winner':<20}")
print("-" * 92)
for f in range(4):
    pos0_score = scores[0, f, 0].item()
    pos1_score = scores[0, f, 1].item()
    win_pos = winning_positions[0, f].item()
    win_words = ' '.join(tokens[win_pos:win_pos+3])
    print(f"F{f:<7} {pos0_score:>6.3f}  '{' '.join(tokens[0:3]):<15}' "
          f"{pos1_score:>6.3f}  '{' '.join(tokens[1:4]):<15}' "
          f"{win_pos}  '{win_words}'")

print(f"\nAfter max-pooling: features = {pooled[0].tolist()}")
print(f"Class logits: {[f'{l:.2f}' for l in logits[0].tolist()]}")
print(f"Predicted class: {tiny_labels[predicted]}  {'✓ correct' if predicted == true_label else '✗ wrong'}")

Tracing sentence: 'My laptop is broken'  (true class: negative_tech)

Filter   Pos 0 (words 0-2)            Pos 1 (words 1-3)            Winner              
--------------------------------------------------------------------------------------------
F0       17.293  'my laptop is   '  4.404  'laptop is broken' 0  'my laptop is'
F1        0.000  'my laptop is   '  0.000  'laptop is broken' 0  'my laptop is'
F2        1.565  'my laptop is   '  3.727  'laptop is broken' 1  'laptop is broken'
F3        0.000  'my laptop is   '  0.138  'laptop is broken' 1  'laptop is broken'

After max-pooling: features = [17.29262924194336, 0.0, 3.727433443069458, 0.1381022334098816]
Class logits: ['-2.33', '8.00', '-13.42', '-0.45']
Predicted class: negative_tech  ✓ correct


Class logits: [-2.33, 8.00, -13.42, -0.45]
              ↓        ↓       ↓         ↓
              negative negative positive positive
              food     tech     food     tech

negative_tech scored 8.00 — massively higher than any other class. How did the classifier get here? Reading roughly:

Filter 0 firing at 17.29 must have a strong positive weight for negative_tech in the classifier
Filter 2 firing at 3.73 likely has a positive weight for negative_tech too (it fires on negative sentiment patterns generally)
Filter 3 near zero provides no positive food signal
positive_food scored -13.42, the lowest — the classifier heavily penalizes any pull toward positive food when tech signal is strong
Key insight for students: the CNN doesn't have a filter that says "negative tech" directly. It has a "tech" filter and a "negative sentiment" filter, and the classifier combines them to produce the right prediction. This is compositionality at work — filters learn primitive features, the classifier composes them into class predictions.

Method 3 will confirm this by literally showing the classifier weights. Let's do it now.

# Method 3: read the classifier weights

In [16]:
# The classifier is Linear(4, 4). Read the weights: which filter votes for which class?
import numpy as np
weights = model.classifier.weight.detach().numpy()   # shape (n_classes, n_filters)
biases = model.classifier.bias.detach().numpy()

print("Classifier weights — rows = classes, columns = filters:")
print(f"{'Class':<20} {'F0':>10} {'F1':>10} {'F2':>10} {'F3':>10} {'Bias':>10}")
print("-" * 72)
for i, cls in enumerate(tiny_labels):
    row = weights[i]
    row_str = ' '.join(f'{w:>10.3f}' for w in row)
    print(f"{cls:<20} {row_str} {biases[i]:>10.3f}")

Classifier weights — rows = classes, columns = filters:
Class                        F0         F1         F2         F3       Bias
------------------------------------------------------------------------
negative_food            -0.296     -0.182      0.903     -0.134     -0.556
negative_tech             0.463      0.072     -0.073     -0.989      0.400
positive_food            -0.738      0.104     -0.227      0.928      0.053
positive_tech             0.170     -0.055     -0.905      0.519     -0.093


Filter 0 (tech detector):

negative_tech: +0.463 (strong positive)
positive_tech: +0.170 (positive but smaller)
negative_food: −0.296 (negative)
positive_food: −0.738 (strong negative)

Perfect tech detector: pushes toward both tech classes, pushes away from both food classes.

Filter 2 (negative food detector):

negative_food: +0.903 (strong positive)
positive_food: −0.227 (negative)
positive_tech: −0.905 (strong negative)
negative_tech: −0.073 (near zero)
More nuanced than expected: Filter 2 strongly inhibits positive_tech. This means when the "negative pattern" fires, the model says "definitely not positive_tech."

Filter 3 (positive food detector):

positive_food: +0.928 (strong positive)
negative_tech: −0.989 (strong negative)
positive_tech: +0.519 (positive)
negative_food: −0.134 (small negative)

Also more nuanced: Filter 3 strongly inhibits negative_tech. Positive vocabulary → not negative_tech.

In [18]:
# Verify: manually recompute the class logits for "My laptop is broken"
features = pooled[0].numpy()
print(f"Features (max-pooled): {features}\n")
print(f"{'Class':<20} {'Manual calculation':<70} {'= logit'}")
print("-" * 100)
for i, cls in enumerate(tiny_labels):
    row = weights[i]
    contribs = [features[j] * row[j] for j in range(4)]
    calc = ' + '.join(f'{features[j]:.2f}×{row[j]:.3f}' for j in range(4)) + f' + {biases[i]:.3f}'
    total = sum(contribs) + biases[i]
    print(f"{cls:<20} {calc:<70} = {total:>+6.2f}")

Features (max-pooled): [17.29263     0.          3.7274334   0.13810223]

Class                Manual calculation                                                     = logit
----------------------------------------------------------------------------------------------------
negative_food        17.29×-0.296 + 0.00×-0.182 + 3.73×0.903 + 0.14×-0.134 + -0.556         =  -2.33
negative_tech        17.29×0.463 + 0.00×0.072 + 3.73×-0.073 + 0.14×-0.989 + 0.400           =  +8.00
positive_food        17.29×-0.738 + 0.00×0.104 + 3.73×-0.227 + 0.14×0.928 + 0.053           = -13.42
positive_tech        17.29×0.170 + 0.00×-0.055 + 3.73×-0.905 + 0.14×0.519 + -0.093          =  -0.45


# Phase 7 wrap-up markdown

### Phase 7 wrap-up — the CNN, fully understood

We opened up a trained neural network and traced every step from input to prediction. Three views of the same model:

**Method 1 — Top-activating n-grams**: showed each filter's "specialty":
- Filter 0: tech topic detector (fires on tech vocabulary regardless of sentiment)
- Filter 1: dead filter (ReLU killed it during training)
- Filter 2: negative food detector (topic × sentiment combined)
- Filter 3: positive food detector (topic × sentiment combined)

**Method 2 — Single-sentence trace**: showed which n-gram won max-pooling for each filter on one specific input.

**Method 3 — Classifier weights + manual math**: showed exactly how the four filter activations combine linearly into class logits. **Every prediction was reproducible by hand.**

### Three takeaways students should carry forward

**1. Neural networks are not black boxes.** They *look* opaque because modern models have millions or billions of parameters. But every single one decomposes to arithmetic like what we just did. The tools for inspection (top-activating inputs, gradient tracing, weight examination) scale to any size.

**2. Filters learn primitive features; classifiers compose them.** Our CNN has no "broken laptop" filter. It has a "tech" filter and a "negative sentiment" filter, and the classifier layer combines them. This compositional structure is why deep learning generalizes — features learned on one class transfer to new combinations.

**3. Dead ReLUs are real.** Filter 1 in our model contributed nothing throughout training and inference. In production, this is a common failure mode fixed with better initialization (Kaiming/He), Leaky ReLU activation, batch normalization, or simply having more filters so the dead ones don't matter.

### Ready to scale

We proved the architecture works on our 48-sentence toy corpus and understand *why* it works. Phase 9 applies the same architecture to real data — 2327 training documents in 20 Newsgroups — and asks: **can a CNN beat TF-IDF and averaged embeddings on the real task?**

# Phase 9 intro: scale up to 20 Newsgroups

---

## Phase 9 — Scale up to 20 Newsgroups

We proved the CNN architecture works on 48 hand-crafted sentences. Now the real test: **can it beat TF-IDF and averaged embeddings on 2327 real Usenet posts?**

### What stays the same

Same 4 categories from Lab 05: `sci.med`, `sci.space`, `rec.autos`, `talk.politics.guns`. Same train/test split. Same evaluation on unseen test documents. This is a **direct apples-to-apples comparison** with Lab 05's four methods.

### What changes

The tiny corpus had 4-word sentences. Real Usenet posts have hundreds of words. Three architectural adjustments:

**1. Multiple filter widths.** The tiny CNN used one filter width (3). Real text has patterns at multiple granularities: 2-word phrases ("not bad"), 3-word phrases ("was a disaster"), 4-word phrases ("has been demonstrated to"). We use widths 3, 4, and 5 in parallel — following Kim (2014).

**2. More filters per width.** 4 filters was enough to cover 4 classes on a toy corpus. Real text has hundreds of relevant patterns per class. We use **100 filters per width** (300 total). Still tiny by modern standards but enough to cover 20 Newsgroups.

**3. Longer input sequences.** Padding to length 400 words. Documents shorter than 400 words get zero-padded; longer ones get truncated. 400 is a sensible middle ground for Usenet posts.

### What we expect

Based on Lab 05's results:
- TF-IDF: 85.16% (the surprisingly strong baseline)
- Averaged Word2Vec: 81.10%
- Averaged GloVe: 80.97%
- Sentence-BERT: 88.45% (the transformer)

**CNN prediction: somewhere between averaged embeddings and SBERT.** Roughly **83-87%** is realistic. If we beat 85% (TF-IDF), that's a real success — a CNN training from scratch on modest hardware, outperforming a strong classical baseline. If we don't quite reach it, that's honest — CNNs are historically outshined by transformers on this task.

Either way, we complete the story: **CNNs bridge the gap between averaging (lost information) and attention (transformers)**.

### One honest caveat

The training will take **2-5 minutes on CPU** (longer than any previous cell). This is our first taste of real deep learning compute cost. Word2Vec lookup is fast; averaging is trivial; training a CNN from scratch on 2327 documents with 300 filters is not. In production, this would run on GPU in seconds. On CPU, we wait.